# Small cells: error bounds, exact certificates, and practical geometry

This tutorial uses the **v0.6.0 development features** of m_scraps. Install a checkout containing `bounds`, `enumerate`, and `search-cells`, with the notebook dependencies (`pip install ".[notebooks]"`). A frozen v0.5.1 installation does not contain these additions.

We distinguish a **necessary lower bound** from an **attainable minimum**, then compare compact cells at fixed compositions. Both single- and multi-sublattice examples are included. All objectives here are explicit **pair-SRO errors**, not decorated pair/triplet/quadruplet correlation errors or native SCRAPS fitness.

No ATAT, icet, or native optimizer is invoked. Lattice parameters are illustrative; these prototypes are not predictions of phase stability. The examples are deliberately small, not a new benchmark campaign.

In [ ]:
import json
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from ase.build import bulk
from ase.io import write
from IPython.display import Markdown, display
from m_scraps import sro_bounds, enumerate_sro, search_cells, analyze_structure

work = Path(mkdtemp(prefix="m_scraps_small_cells_"))
print("Results:", work)

def target(alpha, source="all", destination="all", a="Cu", b="Zn"):
    return dict(source=source, destination=destination, shell=1,
                center_species=a, neighbor_species=b, target=alpha)

def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n| " + " | ".join(["---"] * len(headers)) + " |\n" +
                     "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))


## 1. A small cell can have a large unavoidable error

The two sites in a conventional BCC cell share many periodic-image bonds. For equiatomic Cu–Zn, the random target (alpha = 0) is impossible in this cell. Integer rounding alone misses the restriction; periodic bond multiplicities prove a minimum error of 1.

When the input structure already attains a rigorous bound, `enumerate_sro` certifies it **without enumerating occupations or discovering symmetry**. The initial witness is still evaluated. `complete` records full enumeration coverage, not whether the minimum is proved.

In [ ]:
tiny = bulk("Cu", "bcc", a=3, cubic=True)
tiny[1].symbol = "Zn"
zero_targets = [target(0)]
bound = sro_bounds(tiny, cutoff=2.7, targets=zero_targets)
proof, witness = enumerate_sro(tiny, cutoff=2.7, targets=zero_targets)
assert bound["entries"][0]["integer_only_lower_bound_exact"] == "0"
assert proof["best_error"] == 1
assert proof["certificate_basis"] == "bound_attained"
assert proof["evaluated_configurations"] == 0 and not proof["complete"]
table(["Integer-only bound", "Graph-aware bound", "Attained error", "Enumerated representatives"],
      [[0, bound["result"]["lower_bound"], proof["best_error"], proof["evaluated_configurations"]]])


## 2. A stronger lower bound can still be loose

For a 16-site BCC cell targeting alpha = −0.1, binary cut parity strengthens the bound from 0.00625 to 0.025. Exhaustive symmetry-reduced enumeration establishes a larger minimum, 0.0375.

Thus a requested error of 0.01 can already be ruled out from the bound. To establish the actual minimum, the remaining gap must still be resolved. A zero bound alone is never a feasibility certificate. Use `stop_at_bound=False` (CLI `--exhaustive`) when auditing full coverage rather than seeking the earliest proof.

In [ ]:
bcc16 = bulk("Cu", "bcc", a=3, cubic=True).repeat((2, 2, 2))
bcc16.symbols[8:] = "Zn"
nonzero_targets = [target(-0.1)]
bound16 = sro_bounds(bcc16, cutoff=2.7, targets=nonzero_targets)
exact16, best16 = enumerate_sro(bcc16, cutoff=2.7, targets=nonzero_targets,
                               max_configurations=20000)
assert bound16["result"]["lower_bound"] == 0.025
assert exact16["best_error"] == 0.0375
assert exact16["certificate_basis"] == "exhaustion" and exact16["complete"]
independent = analyze_structure(best16, cutoff=2.7, sro_targets=nonzero_targets)
assert np.isclose(independent["sro_comparison"]["metrics"]["max_error"], exact16["best_error"])
table(["Sites", "Lower bound", "Certified minimum", "Representatives", "Labeled occupations covered"],
      [[16, bound16["result"]["lower_bound"], exact16["best_error"],
        exact16["evaluated_configurations"], exact16["covered_configurations"]]])


## 3. Two independently disordered sublattices

In rocksalt (Ti,Zr)(C,N), metal and anion occupations have separate 50:50 budgets. Labels refer to site order and cannot be exchanged. The explicit target below is the first metal-to-anion shell's Ti–N SRO. Other components are not implicitly targeted.

In [ ]:
rocksalt_parent = bulk("TiC", "rocksalt", a=4.4)
rocksalt = rocksalt_parent.repeat((2, 2, 2))
labels = ["M", "X"] * 8
symbols = rocksalt.get_chemical_symbols()
for i in range(4, 8):
    symbols[2*i] = "Zr"
    symbols[2*i+1] = "N"
rocksalt.set_chemical_symbols(symbols)
config = dict(site_labels=labels, allowed={"M": ["Ti", "Zr"], "X": ["C", "N"]})
rocksalt_targets = [target(-0.1, "M", "X", "Ti", "N")]
rock_bound = sro_bounds(rocksalt, cutoff=2.3, sublattices=config, targets=rocksalt_targets)
rock_proof, rock_witness = enumerate_sro(rocksalt, cutoff=2.3, sublattices=config,
                                       targets=rocksalt_targets, max_configurations=10000)
assert rock_proof["status"] == "certified_minimum"
checked = analyze_structure(rock_witness, cutoff=2.3, sublattices=config, sro_targets=rocksalt_targets)
assert np.isclose(checked["sro_comparison"]["metrics"]["max_error"], rock_proof["best_error"])
table(["System", "Lower bound", "Certified minimum", "Proof"],
      [["(Ti,Zr)(C,N)", rock_bound["result"]["lower_bound"], rock_proof["best_error"],
        rock_proof["certificate_basis"]]])


## 4. Constrain cell geometry before occupation optimization

`max_aspect=1.5` limits the longest/shortest **Minkowski-reduced** basis-vector ratio. `min_repeat=4` requires the shortest periodic lattice translation to be at least 4 angstrom. Reduction removes avoidable basis skew, while the filters exclude intrinsically thin or elongated cells. These are geometry choices, not a guarantee of DFT convergence or orthogonality.

We examine at most four candidates per system, across 4–16 sites. Small occupation spaces use exact enumeration; larger ones use a seeded Python pair-SRO SA (distinct from the native SCRAPS solver). Step budgets and seeds are reproducible; wall-time truncation can alter results. Bounds and errors use the same explicit targets and metric.

In [ ]:
parent_config = dict(site_labels=["M", "X"], allowed=config["allowed"])
cases = [
    ("BCC", bulk("Cu", "bcc", a=3, cubic=True), 2.7, nonzero_targets, None,
     {"all": {"Cu": "1/2", "Zn": "1/2"}}),
    ("rocksalt", rocksalt_parent, 2.3, rocksalt_targets, parent_config,
     {"M": {"Ti": "1/2", "Zr": "1/2"}, "X": {"C": "1/2", "N": "1/2"}}),
]
rows = []
search_reports = {}
for name, parent, cutoff, targets, sublattices, composition in cases:
    report, witnesses = search_cells(parent, cutoff=cutoff, targets=targets,
        sublattices=sublattices, composition=composition, min_atoms=4, max_atoms=16,
        min_repeat=4, max_aspect=1.5, max_candidates=4, max_hnfs=500,
        max_configurations=1000, sa_steps=300, seed=42, max_seconds=30, seconds_per_cell=5)
    assert report["candidates"] and not report["global_optimum_certified"]
    directory = work / name
    directory.mkdir()
    for candidate in report["candidates"]:
        geometry = candidate["geometry"]
        assert geometry["aspect_ratio"] <= 1.5 + 1e-10
        assert geometry["shortest_repeat"] >= 4 - 1e-10
        structure = witnesses[candidate["id"]]
        check = analyze_structure(structure, cutoff=cutoff,
            sublattices=candidate["sublattices"], sro_targets=targets)
        assert np.isclose(check["sro_comparison"]["metrics"]["max_error"], candidate["best_error"])
        write(directory / (candidate["id"] + ".extxyz"), structure)
        rows.append([name, candidate["atoms"], f'{candidate["best_error"]:.6f}',
                     f'{geometry["aspect_ratio"]:.3f}', f'{geometry["shortest_repeat"]:.3f}',
                     candidate["status"], candidate.get("pareto", False)])
    (directory / "result.json").write_text(json.dumps(report, indent=2) + "\n")
    search_reports[name] = report
table(["System", "Atoms", "Error", "Aspect", "Repeat (Å)", "Status", "Pareto"], rows)


## 5. Skip cells that cannot meet the requested tolerance

`error_tolerance` screens candidates using a rigorous lower bound; it does not confuse that bound with the attainable minimum. The 16-site BCC candidates below cannot reach error ≤0.01, so no occupation search is launched.

In [ ]:
screened, witnesses = search_cells(bulk("Cu", "bcc", a=3, cubic=True),
    cutoff=2.7, targets=nonzero_targets, composition={"all": {"Cu": "1/2", "Zn": "1/2"}},
    min_atoms=16, max_atoms=16, max_candidates=4, max_hnfs=500,
    max_aspect=1.5, min_repeat=4, error_tolerance=0.01)
assert screened["candidates"] and not witnesses
assert all(c["status"] == "screened_by_bound" for c in screened["candidates"])
print("Cells screened without occupation optimization:", len(screened["candidates"]))
(work / "certificates.json").write_text(json.dumps(
    dict(bcc2=proof, bcc16=exact16, rocksalt=rock_proof, screening=screened), indent=2) + "\n")


## Interpreting the result

- **Bound:** no allowed occupation of the specified cell can have a smaller selected error. Joint constraints can make the actual minimum higher.
- **Certificate:** an exact bound match or exhaustive enumeration proves the selected minimum for one specified cell, composition, labels, and target set.
- **Shape search:** geometry filters, HNF/candidate caps, and finite occupation budgets restrict the explored space. It does not certify a global optimum across sizes and shapes.
- **Pareto candidates:** compare achieved errors, atom counts, aspect ratios, and shortest repeats among evaluated cells. They do not compare unknown true minima.

Only explicit pair-SRO targets contribute, with equal weights and uniform coordination required in each requested sublattice-pair shell. Bounds do **not** cover decorated higher-order correlations. A small cell is useful only if its accuracy and geometry both suit the intended downstream calculation.

CLI equivalents: `m_scraps bounds`, `m_scraps enumerate [--exhaustive]`, and `m_scraps search-cells --max-aspect 1.5 --min-repeat 4 --error-tolerance 0.01` (plus the structure, target, cutoff, size-range and output arguments).